In [139]:
import pandas as pd
import re

In [140]:
kagglepath =  "../../dataset/kgdataset.csv"
surveypath = "../../dataset/survey_data.xlsx"
df = pd.read_csv(kagglepath)
df2 = pd.read_excel(surveypath)
df2.drop(df2.columns[0],axis=1, inplace=True)
df2.columns = df.columns
combined = pd.concat([df, df2], ignore_index=True)
combined


,Age,Sleep Hours,Feel Rested,Daily Screen Time,Use Before Sleep,Stress Level,Anxiety/Low Mood,Wellness Apps,Sleep Quality,Screen Time Affects Sleep?
0,47,4,Yes,11,Yes,10,No,No,Bad,Yes
1,48,4,No,8,No,7,Yes,Yes,Bad,Yes
2,40,7,Sometimes,10,No,4,No,Yes,Bad,Yes
3,26,10,Yes,5,Yes,1,Yes,No,Good,No
4,47,5,Yes,7,Yes,7,Yes,No,Good,Yes
...,...,...,...,...,...,...,...,...,...,...
1181,21,10,สดชื่น,8,ใช่,4,ไม่หงุดหงิด/ไม่วิตกกังวล,ใช่,ดี,ไม่แน่ใจ
1182,17,7-8,ไม่สดชื่น,>8ชม.,ใช่,8,ไม่หงุดหงิด/ไม่วิตกกังวล,ไม่ใช่,ดี,ไม่แน่ใจ
1183,17,6-8,สดชื่น,3-4,ใช่,6,ไม่หงุดหงิด/ไม่วิตกกังวล,ใช่,ไม่ดี,ใช่
1184,17,6,ไม่สดชื่น,4,ใช่,6,ไม่หงุดหงิด/ไม่วิตกกังวล,ไม่ใช่,ไม่ดี,ใช่


In [141]:
combined["Feel Rested"] = combined["Feel Rested"].replace("ไม่สดชื่น", "No").replace("สดชื่น", "Yes")
combined["Use Before Sleep"] = combined["Use Before Sleep"].replace("ไม่ใช้", "No").replace("ใช่", "Yes")
combined["Anxiety/Low Mood"] = combined["Anxiety/Low Mood"].replace("ไม่หงุดหงิด/ไม่วิตกกังวล", "No").replace("หงุดหงิด/วิตกกังวล", "Yes")
combined["Wellness Apps"] = combined["Wellness Apps"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
combined["Sleep Quality"] = combined["Sleep Quality"].replace("ไม่ดี", "Bad").replace("ดี", "Good")
combined["Screen Time Affects Sleep?"] = combined["Screen Time Affects Sleep?"].replace("ไม่แน่ใจ", "Not Sure").replace("ใช่", "Yes").replace("ไม่ใช่", "No")


In [142]:
print(combined.isnull().sum())

Age                           0
Sleep Hours                   0
Feel Rested                   0
Daily Screen Time             0
Use Before Sleep              0
Stress Level                  0
Anxiety/Low Mood              0
Wellness Apps                 0
Sleep Quality                 0
Screen Time Affects Sleep?    0
dtype: int64


In [143]:
#clean age
def extract_number(value):
    if "-" in str(value):
        value = re.sub(r'[^\d\-\.]', '', value)
        values = str(value).split("-")
        return f"{values[0]}-{values[1]}"
    elif isinstance(value, str):
        result = pd.Series(value).str.extract('(\d+)') 
        return result[0].iloc[0] if not result.empty else None

    else:
        return str(value)      
combined["Age"] = combined["Age"].apply(extract_number)

#clean sleep hours by mean
def clean_range_with_mean(value):
    if isinstance(value, str):
        if '-' in value:
            try:
                low, high = map(float, value .split('-'))
                return (low + high) / 2
            except ValueError:
                return value
        else:
            try:
                return float(value)
            except ValueError:
                return value
    else:
        return value
combined["Sleep Hours"] = combined["Sleep Hours"].apply(extract_number)
combined["Sleep Hours"] = combined["Sleep Hours"].apply(clean_range_with_mean)

#Clean Daily Screen Time
combined["Daily Screen Time"] = combined["Daily Screen Time"].apply(extract_number)
combined["Daily Screen Time"] = combined["Daily Screen Time"].apply(clean_range_with_mean)
print(combined["Sleep Hours"].isnull().sum())
print(combined["Age"].isnull().sum())
print(combined["Daily Screen Time"].isnull().sum())


0
0
0


In [144]:
combined.tail(87)

,Age,Sleep Hours,Feel Rested,Daily Screen Time,Use Before Sleep,Stress Level,Anxiety/Low Mood,Wellness Apps,Sleep Quality,Screen Time Affects Sleep?
1099,17,9.0,Sometimes,4.0,Yes,5,No,No,Good,No
1100,20,3.0,No,14.0,Yes,5,No,No,Bad,Not Sure
1101,20,6.0,No,7.0,Yes,6,No,No,Good,Yes
1102,23,10.0,Yes,14.0,Yes,3,No,No,Good,Yes
1103,21,6.0,No,9.0,Yes,7,No,No,Bad,Yes
...,...,...,...,...,...,...,...,...,...,...
1181,21,10.0,Yes,8.0,Yes,4,No,Yes,Good,Not Sure
1182,17,7.5,No,8.0,Yes,8,No,No,Good,Not Sure
1183,17,7.0,Yes,3.5,Yes,6,No,Yes,Bad,Yes
1184,17,6.0,No,4.0,Yes,6,No,No,Bad,Yes
